In [2]:
import sys
sys.path.append('../')

import math
from itertools import product
import pandas as pd
import numpy as np
from src.models import BM25
from src.PreProcessingPipeline import BM25_PreProcess


In [3]:
import sys
sys.path.append("../")

import math
import importlib
import numpy as np
import pandas as pd
from itertools import product

import src.query_data
importlib.reload(src.query_data)

from src.query_data import corpus_df, corpus, processed_corpus, query, query_prepro
from src.models import BM25

Columns: ['category', 'filename', 'title', 'content']


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\charl\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\charl\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Columns: ['category', 'filename', 'title', 'content']


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\charl\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\charl\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [9]:
assert len(corpus) == len(processed_corpus), "Corpus and processed_corpus lengths do not match"
assert len(query) == len(query_prepro), "Query and query_prepro lengths do not match"
assert "category" in corpus_df.columns, "corpus_df must contain a category column"
assert "category" in query.columns, "query must contain a category column"

print("Documents:", len(corpus))
print("Queries:", len(query))

Documents: 2225
Queries: 25


In [5]:
def average_precision(relevance):
    """
    relevance: list of 0/1 values in ranked order
    """
    num_rel = sum(relevance)
    if num_rel == 0:
        return 0.0

    precisions = []
    rel_found = 0

    for i, rel in enumerate(relevance, start=1):
        if rel == 1:
            rel_found += 1
            precisions.append(rel_found / i)

    return sum(precisions) / num_rel


def precision_at_k(relevance, k=10):
    if len(relevance) == 0:
        return 0.0
    return sum(relevance[:k]) / k


def dcg_at_k(relevance, k=10):
    relevance = relevance[:k]
    return sum(rel / math.log2(i + 2) for i, rel in enumerate(relevance))


def ndcg_at_k(relevance, k=10):
    dcg = dcg_at_k(relevance, k)
    ideal = sorted(relevance, reverse=True)
    idcg = dcg_at_k(ideal, k)
    return dcg / idcg if idcg > 0 else 0.0

In [ ]:
# change code to rank results by relevance instead of category match
# create code to store results in a csv dataframe for each query with columns: query, target_category, AP, Precision@10, nDCG@10 and show the top 5 queries & category with the highest AP scores
# create code to state the best performing k1 and b values based on the evaluation results


In [6]:
def evaluate_bm25_model(
    bm25,
    query_df,
    query_prepro_list,
    ndcg_k=10,
    max_queries=None
):
    ap_scores = []
    p10_scores = []
    ndcg_scores = []
    per_query_results = []

    n_queries = len(query_df) if max_queries is None else min(max_queries, len(query_df))

    for i in range(n_queries):
        row = query_df.iloc[i]
        raw_query = row["query"]
        relevant_category = row["category"]

        ranked_results = bm25.rank(query_prepro_list[i])

        relevance = ranked_results["category"].eq(relevant_category).astype(int).tolist()

        ap = average_precision(relevance)
        p10 = precision_at_k(relevance, k=10)
        ndcg = ndcg_at_k(relevance, k=ndcg_k)

        ap_scores.append(ap)
        p10_scores.append(p10)
        ndcg_scores.append(ndcg)

        per_query_results.append({
            "query": raw_query,
            "category": relevant_category,
            "AP": ap,
            "Precision@10": p10,
            f"nDCG@{ndcg_k}": ndcg
        })

    summary = {
        "MAP": round(float(np.mean(ap_scores)), 4),
        "Precision@10": round(float(np.mean(p10_scores)), 4),
        f"nDCG@{ndcg_k}": round(float(np.mean(ndcg_scores)), 4)
    }

    per_query_df = pd.DataFrame(per_query_results)
    return summary, per_query_df

Evaluation is done in batches as it takes long

In [8]:
from itertools import product
import pandas as pd
import os

k1_values = [0.5]
b_values = [0.25]

all_results = []

for k1, b in product(k1_values, b_values):
    bm25 = BM25(corpus=processed_corpus, k1=k1, b=b)

    summary_test, per_query_test = evaluate_bm25_model(
        bm25=bm25,
        query_df=query,
        query_prepro_list=query_prepro,
        ndcg_k=10,
        max_queries=None
    )

    queries_ranked = per_query_test.sort_values("AP", ascending=False).reset_index(drop=True)
    top_5_queries = queries_ranked.head(5).copy()

    for _, row in top_5_queries.iterrows():
        all_results.append({
            "k1": k1,
            "b": b,
            "MAP": summary_test["MAP"],
            "Precision@10_overall": summary_test["Precision@10"],
            "nDCG@10_overall": summary_test["nDCG@10"],
            "query": row["query"],
            "AP": row["AP"],
            "Precision@10_query": row["Precision@10"],
            "nDCG@10_query": row["nDCG@10"]
        })

results_df = pd.DataFrame(all_results)
results_df.drop_duplicates(inplace=True)

file_path = "../data/archive-5/bm25_evaluation_results_with_top5_queries.csv"

# If file exists → append, otherwise create with header
if os.path.exists(file_path):
    existing_df = pd.read_csv(file_path)
    combined_df = pd.concat([existing_df, results_df], ignore_index=True)

    combined_df.drop_duplicates(subset=["k1", "b", "query"], inplace=True)
    combined_df.to_csv(file_path, index=False)
else:
    results_df.to_csv(file_path, index=False)

print("Saved (appended if exists): bm25_evaluation_results_with_top5_queries.csv")

Saved (appended if exists): bm25_evaluation_results_with_top5_queries.csv


In [ ]:
from itertools import product
import pandas as pd
import os

k1_values = [0.5]
b_values = [0.5, 0.75, 1.0]

all_results = []

for k1, b in product(k1_values, b_values):
    bm25 = BM25(corpus=processed_corpus, k1=k1, b=b)

    summary_test, per_query_test = evaluate_bm25_model(
        bm25=bm25,
        query_df=query,
        query_prepro_list=query_prepro,
        ndcg_k=10,
        max_queries=None
    )

    queries_ranked = per_query_test.sort_values("AP", ascending=False).reset_index(drop=True)
    top_5_queries = queries_ranked.head(5).copy()

    for _, row in top_5_queries.iterrows():
        all_results.append({
            "k1": k1,
            "b": b,
            "MAP": summary_test["MAP"],
            "Precision@10_overall": summary_test["Precision@10"],
            "nDCG@10_overall": summary_test["nDCG@10"],
            "query": row["query"],
            "AP": row["AP"],
            "Precision@10_query": row["Precision@10"],
            "nDCG@10_query": row["nDCG@10"]
        })

results_df = pd.DataFrame(all_results)
results_df.drop_duplicates(inplace=True)

file_path = "../data/archive-5/bm25_evaluation_results_with_top5_queries.csv"

# If file exists → append, otherwise create with header
if os.path.exists(file_path):
    existing_df = pd.read_csv(file_path)
    combined_df = pd.concat([existing_df, results_df], ignore_index=True)

    combined_df.drop_duplicates(subset=["k1", "b", "query"], inplace=True)
    combined_df.to_csv(file_path, index=False)
else:
    results_df.to_csv(file_path, index=False)

print("Saved (appended if exists): bm25_evaluation_results_with_top5_queries.csv")

In [ ]:
from itertools import product
import pandas as pd
import os

k1_values = [0.5, 1.0, 1.5, 2.0]
b_values = [0.25, 0.5, 0.75, 1.0]

all_results = []

for k1, b in product(k1_values, b_values):
    bm25 = BM25(corpus=processed_corpus, k1=k1, b=b)

    summary_test, per_query_test = evaluate_bm25_model(
        bm25=bm25,
        query_df=query,
        query_prepro_list=query_prepro,
        ndcg_k=10,
        max_queries=None
    )

    queries_ranked = per_query_test.sort_values("AP", ascending=False).reset_index(drop=True)
    top_5_queries = queries_ranked.head(5).copy()

    for _, row in top_5_queries.iterrows():
        all_results.append({
            "k1": k1,
            "b": b,
            "MAP": summary_test["MAP"],
            "Precision@10_overall": summary_test["Precision@10"],
            "nDCG@10_overall": summary_test["nDCG@10"],
            "query": row["query"],
            "AP": row["AP"],
            "Precision@10_query": row["Precision@10"],
            "nDCG@10_query": row["nDCG@10"]
        })

results_df = pd.DataFrame(all_results)
results_df.drop_duplicates(inplace=True)

file_path = "../data/archive-5/bm25_evaluation_results_with_top5_queries.csv"

# If file exists → append, otherwise create with header
if os.path.exists(file_path):
    existing_df = pd.read_csv(file_path)
    combined_df = pd.concat([existing_df, results_df], ignore_index=True)

    combined_df.drop_duplicates(subset=["k1", "b", "query"], inplace=True)
    combined_df.to_csv(file_path, index=False)
else:
    results_df.to_csv(file_path, index=False)

print("Saved (appended if exists): bm25_evaluation_results_with_top5_queries.csv")

Saved (appended if exists): bm25_evaluation_results_with_top5_queries.csv
